# Week 5: CSV/JSON — Reading the World — PHASE 3: Data Tools

*Core Mastery: "I can read, validate, and write structured data in CSV and JSON formats"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Explain why structured file formats (CSV, JSON) are essential for data pipelines
2. Read CSV data using `csv.reader()` and `csv.DictReader()` with `io.StringIO`
3. Write CSV files using `csv.writer()` and `csv.DictWriter()`
4. Parse JSON strings with `json.loads()` and JSON files with `json.load()`
5. Serialize Python objects to JSON with `json.dumps()` and `json.dump()` (including Turkish characters)
6. Define a data schema as a dictionary specifying column names, types, and valid ranges
7. Implement a `validate_schema()` function that checks data against a schema
8. Build a complete Read → Validate → Transform → Write pipeline
9. Use `io.StringIO` to simulate file I/O in Colab without creating actual files
10. Handle common data format issues: missing headers, encoding, delimiters

## 🎯 Core Mastery Connection

Last week you learned to use dictionaries as pipeline tools — counting, bucketing, and looking up data. But where does that data **come from**, and where does it **go**? In the real world, data lives in **files**: CSV spreadsheets exported from sensors, JSON responses from web APIs, configuration files for equipment.

This week you learn to **read** data from these standard formats into Python data structures (lists and dicts), **validate** that the data matches your expectations, and **write** processed results back out. This completes the data-flow story: file → Python → process → Python → file.

---
## Part 1: Why Structured Data?

Every data pipeline needs a way to **receive input** and **produce output**. Raw text is too unstructured — we need formats with clear rules.

| Feature | CSV | JSON |
|---------|-----|------|
| Structure | Rows and columns (tabular) | Nested key-value (hierarchical) |
| Human readable | Yes (open in Excel) | Yes (open in text editor) |
| Best for | Tables, sensor logs, grades | API responses, configs, nested data |
| Python module | `csv` | `json` |
| Delimiter | Comma (or `;`, `\t`) | None (self-describing) |
| Data types | Everything is a string | Strings, numbers, bools, null, arrays, objects |
| Nested data | Not supported | Native support |

**CSV example:**
```
sensor_id,temperature,status
S01,23.5,OK
S02,45.2,WARN
```

**JSON example:**
```json
[
  {"sensor_id": "S01", "temperature": 23.5, "status": "OK"},
  {"sensor_id": "S02", "temperature": 45.2, "status": "WARN"}
]
```

We will use `io.StringIO` throughout this notebook so all examples run in Colab without needing actual files on disk.

---
## Part 2: Reading CSV — `csv.reader()` and `csv.DictReader()`

The `csv` module provides two readers:
- **`csv.reader()`** — returns each row as a list of strings
- **`csv.DictReader()`** — returns each row as a dictionary (column name → value)

We use `io.StringIO(text)` to create a file-like object from a string.

**Figure 2.1** — Reading CSV with `csv.reader()`

In [ ]:
import csv
import io

csv_text = """sensor_id,temperature,humidity,status
S01,23.5,55,OK
S02,45.2,30,WARN
S03,19.8,62,OK
S04,38.7,41,CRITICAL
S05,22.1,58,OK"""

reader = csv.reader(io.StringIO(csv_text))
header = next(reader)  # First row is the header
print("Header:", header)
print("-" * 40)

rows = []
for row in reader:
    print(row)
    rows.append(row)

print(f"\nTotal data rows: {len(rows)}")

**Figure 2.2** — Reading CSV with `csv.DictReader()`

In [ ]:
import csv, io

csv_text = """student_id,name,dept,midterm,final
20230101,Zeynep Kaya,EE,78,85
20230102,Ali Demir,ME,65,72
20230103,Fatma Celik,CS,92,95
20230104,Burak Ozkan,EE,70,68
20230105,Elif Yildiz,ME,88,91"""

reader = csv.DictReader(io.StringIO(csv_text))
print("Fieldnames:", reader.fieldnames)
print("-" * 50)

students = []
for row in reader:
    # Note: all values are strings — convert numbers manually
    row["midterm"] = int(row["midterm"])
    row["final"] = int(row["final"])
    row["average"] = row["midterm"] * 0.4 + row["final"] * 0.6
    students.append(row)
    print(f"  {row['name']:<16} {row['dept']}  avg={row['average']:.1f}")

**Figure 2.3** — Handling different delimiters

In [ ]:
import csv, io

# Semicolon-separated data (common in Turkish Excel exports)
csv_text = """ad;soyad;bolum;not
Ahmet;Yilmaz;EE;85
Mehmet;Kara;ME;72
Ayse;Demir;CS;91"""

reader = csv.DictReader(io.StringIO(csv_text), delimiter=";")
print("Fields:", reader.fieldnames)
for row in reader:
    print(f"  {row['ad']} {row['soyad']} — {row['bolum']} — {row['not']}")

---
## Part 3: Writing CSV — `csv.writer()` and `csv.DictWriter()`

Writing CSV reverses the reading process:
- **`csv.writer()`** — writes lists as rows
- **`csv.DictWriter()`** — writes dicts as rows, needs `fieldnames`

**Figure 3.1** — Writing CSV with `csv.writer()`

In [ ]:
import csv, io

output = io.StringIO()
writer = csv.writer(output)

# Write header
writer.writerow(["sensor_id", "temperature", "status"])

# Write data rows
data = [
    ["S01", 23.5, "OK"],
    ["S02", 45.2, "WARN"],
    ["S03", 19.8, "OK"],
]
writer.writerows(data)

# Get the CSV string
csv_result = output.getvalue()
print("Generated CSV:")
print(csv_result)

**Figure 3.2** — Writing CSV with `csv.DictWriter()`

In [ ]:
import csv, io

fields = ["name", "dept", "score", "grade"]
output = io.StringIO()
writer = csv.DictWriter(output, fieldnames=fields)
writer.writeheader()

students = [
    {"name": "Zeynep Kaya",  "dept": "EE", "score": 88, "grade": "B+"},
    {"name": "Ali Demir",    "dept": "ME", "score": 72, "grade": "C+"},
    {"name": "Fatma Celik",  "dept": "CS", "score": 95, "grade": "A"},
]
writer.writerows(students)

print("Generated CSV:")
print(output.getvalue())

**Figure 3.3** — Round-trip: read, transform, write

In [ ]:
import csv, io

# Input CSV
input_csv = """sensor_id,temp_f,status
S01,74.3,OK
S02,113.36,WARN
S03,67.64,OK"""

# Read
reader = csv.DictReader(io.StringIO(input_csv))
processed = []
for row in reader:
    temp_c = round((float(row["temp_f"]) - 32) * 5/9, 1)
    processed.append({
        "sensor_id": row["sensor_id"],
        "temp_c": temp_c,
        "temp_f": row["temp_f"],
        "status": row["status"]
    })

# Write
output = io.StringIO()
writer = csv.DictWriter(output, fieldnames=["sensor_id", "temp_c", "temp_f", "status"])
writer.writeheader()
writer.writerows(processed)

print("Transformed CSV (with Celsius):")
print(output.getvalue())

---
## Part 4: Reading JSON — `json.loads()` and `json.load()`

JSON (JavaScript Object Notation) maps directly to Python:

| JSON | Python |
|------|--------|
| `object {}` | `dict` |
| `array []` | `list` |
| `string` | `str` |
| `number` | `int` or `float` |
| `true/false` | `True/False` |
| `null` | `None` |

- **`json.loads(string)`** — parse a JSON **s**tring
- **`json.load(file)`** — parse a JSON **file** object

**Figure 4.1** — Parsing JSON strings with `json.loads()`

In [ ]:
import json

json_text = '''[
  {"sensor_id": "S01", "temperature": 23.5, "status": "OK"},
  {"sensor_id": "S02", "temperature": 45.2, "status": "WARN"},
  {"sensor_id": "S03", "temperature": 19.8, "status": "OK"}
]'''

data = json.loads(json_text)
print("Type:", type(data))
print("Count:", len(data))
print()
for item in data:
    print(f"  {item['sensor_id']}: {item['temperature']}°C [{item['status']}]")

**Figure 4.2** — Nested JSON structures

In [ ]:
import json

json_text = '''{
  "lab": "Elektronik Lab A",
  "building": "Muhendislik Fakultesi",
  "sensors": [
    {"id": "S01", "type": "temperature", "readings": [22.1, 22.5, 23.0]},
    {"id": "S02", "type": "humidity",    "readings": [55, 58, 52]},
    {"id": "S03", "type": "pressure",    "readings": [1013, 1012, 1014]}
  ],
  "last_update": "2025-03-15T10:30:00"
}'''

lab = json.loads(json_text)
print(f"Lab: {lab['lab']}")
print(f"Building: {lab['building']}")
print(f"Last update: {lab['last_update']}")
print(f"\nSensors ({len(lab['sensors'])}):")
for s in lab["sensors"]:
    avg = sum(s["readings"]) / len(s["readings"])
    print(f"  {s['id']} ({s['type']}): avg={avg:.1f}")

**Figure 4.3** — Reading JSON from a file-like object

In [ ]:
import json, io

# Simulate reading from a file using StringIO
file_content = '{"name": "Mehmet Kara", "dept": "ME", "grades": [78, 85, 90]}'
fake_file = io.StringIO(file_content)

data = json.load(fake_file)  # json.load() for file objects
print("Name:", data["name"])
print("Dept:", data["dept"])
print("Grades:", data["grades"])
print("Average:", sum(data["grades"]) / len(data["grades"]))

---
## Part 5: Writing JSON — `json.dumps()` and `json.dump()`

- **`json.dumps(obj)`** — serialize to a JSON **s**tring
- **`json.dump(obj, file)`** — serialize to a JSON **file**

Key parameters:
- `indent=2` — pretty-print with indentation
- `ensure_ascii=False` — preserve Turkish characters (ç, ş, ğ, ı, ö, ü)

**Figure 5.1** — Basic JSON serialization

In [ ]:
import json

sensor_report = {
    "timestamp": "2025-03-15T14:00:00",
    "sensor_count": 3,
    "readings": [
        {"id": "S01", "temp": 23.5, "ok": True},
        {"id": "S02", "temp": 45.2, "ok": False},
        {"id": "S03", "temp": 19.8, "ok": True},
    ],
    "notes": None
}

# Compact
print("Compact:")
print(json.dumps(sensor_report))

# Pretty
print("\nPretty:")
print(json.dumps(sensor_report, indent=2))

**Figure 5.2** — Turkish characters with `ensure_ascii=False`

In [ ]:
import json

ogrenciler = [
    {"ad": "Ayşe", "soyad": "Çelik", "bölüm": "Bilgisayar Mühendisliği", "not": 92},
    {"ad": "Ömer", "soyad": "Güneş", "bölüm": "Elektrik-Elektronik", "not": 85},
    {"ad": "İrem", "soyad": "Şahin", "bölüm": "İnşaat Mühendisliği", "not": 78},
]

# Without ensure_ascii=False (Turkish chars are escaped)
print("With ASCII escaping:")
print(json.dumps(ogrenciler[0], ensure_ascii=True))

# With ensure_ascii=False (Turkish chars preserved)
print("\nWithout ASCII escaping:")
print(json.dumps(ogrenciler, indent=2, ensure_ascii=False))

**Figure 5.3** — Writing JSON to a file-like object

In [ ]:
import json, io

data = {
    "project": "Sensor Network",
    "version": 2,
    "sensors": ["S01", "S02", "S03"],
    "config": {"interval_sec": 30, "threshold": 40.0}
}

# Write to a StringIO (simulates file)
output = io.StringIO()
json.dump(data, output, indent=2)

# Read back what was written
result = output.getvalue()
print("Written JSON:")
print(result)
print(f"\nJSON length: {len(result)} characters")

---
## Part 6: Data Schema — Defining Expected Structure

A **schema** defines what valid data looks like:
- Which columns/fields are required
- What data types each field should have
- What value ranges are acceptable

We represent schemas as dictionaries for easy validation.

**Figure 6.1** — Defining a schema dictionary

In [ ]:
# Schema for sensor readings
SENSOR_SCHEMA = {
    "fields": {
        "sensor_id": {"type": str,   "required": True,  "pattern": "S\\d{2}"},
        "temperature": {"type": float, "required": True,  "min": -40, "max": 80},
        "humidity":    {"type": float, "required": False, "min": 0,   "max": 100},
        "status":      {"type": str,   "required": True,  "allowed": ["OK", "WARN", "CRITICAL"]},
    }
}

print("Schema fields:")
for field, rules in SENSOR_SCHEMA["fields"].items():
    req = "required" if rules["required"] else "optional"
    print(f"  {field:<14} type={rules['type'].__name__:<6} ({req})")

**Figure 6.2** — Schema for student records

In [ ]:
STUDENT_SCHEMA = {
    "fields": {
        "student_id": {"type": str,   "required": True},
        "name":       {"type": str,   "required": True,  "min_length": 3},
        "dept":       {"type": str,   "required": True,  "allowed": ["EE", "ME", "CE", "CS", "IE"]},
        "midterm":    {"type": float, "required": True,  "min": 0, "max": 100},
        "final":      {"type": float, "required": True,  "min": 0, "max": 100},
    }
}

# Example valid and invalid records
valid_record   = {"student_id": "20230101", "name": "Zeynep Kaya", "dept": "EE", "midterm": 78, "final": 85}
invalid_record = {"student_id": "20230102", "name": "A", "dept": "XX", "midterm": 150, "final": -5}

print("Valid record:", valid_record)
print("Invalid record:", invalid_record)
print("\n(We will validate these in the next part!)")

---
## Part 7: Schema Validation — Checking Data Quality

Now we build a function that checks each record against a schema and reports errors.

**Figure 7.1** — `validate_record()` function

In [ ]:
def validate_record(record, schema):
    """Validate a single record against a schema. Returns list of errors."""
    errors = []
    for field, rules in schema["fields"].items():
        # Check required fields
        if field not in record:
            if rules["required"]:
                errors.append(f"Missing required field: {field}")
            continue

        value = record[field]

        # Check type (try to convert strings to expected type)
        expected_type = rules["type"]
        if expected_type == float and isinstance(value, (int, float)):
            value = float(value)
        elif expected_type == float and isinstance(value, str):
            try:
                value = float(value)
                record[field] = value  # Update with converted value
            except ValueError:
                errors.append(f"{field}: cannot convert '{value}' to float")
                continue

        # Check min/max for numbers
        if isinstance(value, (int, float)):
            if "min" in rules and value < rules["min"]:
                errors.append(f"{field}: {value} < min({rules['min']})")
            if "max" in rules and value > rules["max"]:
                errors.append(f"{field}: {value} > max({rules['max']})")

        # Check allowed values
        if "allowed" in rules and value not in rules["allowed"]:
            errors.append(f"{field}: '{value}' not in {rules['allowed']}")

        # Check min_length for strings
        if "min_length" in rules and isinstance(value, str):
            if len(value) < rules["min_length"]:
                errors.append(f"{field}: length {len(value)} < min_length({rules['min_length']})")

    return errors

# Test with student schema
SCHEMA = {
    "fields": {
        "name": {"type": str, "required": True, "min_length": 3},
        "dept": {"type": str, "required": True, "allowed": ["EE", "ME", "CS"]},
        "score": {"type": float, "required": True, "min": 0, "max": 100},
    }
}

test_records = [
    {"name": "Zeynep Kaya", "dept": "EE", "score": 88},
    {"name": "A",           "dept": "XX", "score": 150},
    {"name": "Burak Oz",    "dept": "CS"},
]

for i, rec in enumerate(test_records):
    errs = validate_record(rec, SCHEMA)
    if errs:
        print(f"Record {i+1} — INVALID:")
        for e in errs:
            print(f"  ❌ {e}")
    else:
        print(f"Record {i+1} — ✅ Valid")

**Figure 7.2** — Validating CSV data against a schema

In [ ]:
import csv, io

csv_text = """sensor_id,temperature,status
S01,23.5,OK
S02,abc,WARN
S03,19.8,INVALID
S04,-50.0,OK
S05,22.1,OK"""

SENSOR_SCHEMA = {
    "fields": {
        "sensor_id":    {"type": str,   "required": True},
        "temperature":  {"type": float, "required": True, "min": -40, "max": 80},
        "status":       {"type": str,   "required": True, "allowed": ["OK", "WARN", "CRITICAL"]},
    }
}

reader = csv.DictReader(io.StringIO(csv_text))
valid_rows = []
invalid_rows = []

for row in reader:
    errors = validate_record(row, SENSOR_SCHEMA)
    if errors:
        invalid_rows.append((row, errors))
    else:
        valid_rows.append(row)

print(f"Valid: {len(valid_rows)}, Invalid: {len(invalid_rows)}")
print("\nInvalid rows:")
for row, errs in invalid_rows:
    print(f"  {row['sensor_id']}: {errs}")

**Figure 7.3** — Validation summary report

In [ ]:
import csv, io

csv_text = """name,dept,midterm,final
Zeynep Kaya,EE,78,85
Ali Demir,ME,65,72
Bad Record,XX,150,-5
Elif Y,CS,88,91
No Dept,,70,80
Fatma Celik,EE,abc,90"""

SCHEMA = {
    "fields": {
        "name":    {"type": str,   "required": True, "min_length": 3},
        "dept":    {"type": str,   "required": True, "allowed": ["EE", "ME", "CS", "CE", "IE"]},
        "midterm": {"type": float, "required": True, "min": 0, "max": 100},
        "final":   {"type": float, "required": True, "min": 0, "max": 100},
    }
}

reader = csv.DictReader(io.StringIO(csv_text))
error_summary = {}

for i, row in enumerate(reader, 1):
    errors = validate_record(row, SCHEMA)
    for e in errors:
        field = e.split(":")[0]
        error_summary[field] = error_summary.get(field, 0) + 1

print("Validation Error Summary:")
print("-" * 30)
for field, count in sorted(error_summary.items(), key=lambda x: -x[1]):
    print(f"  {field:<12} {count} error(s)")

---
## Part 8: Read → Validate → Transform → Write Pipeline

Now we chain everything into a complete pipeline:
1. **Read** CSV input
2. **Validate** each row against a schema
3. **Transform** valid data (compute averages, convert units, etc.)
4. **Write** results as CSV and/or JSON

**Figure 8.1** — Complete pipeline: sensor data processing

In [ ]:
import csv, json, io

# ── STEP 1: READ ──
input_csv = """sensor_id,temp_f,humidity,status
S01,74.3,55,OK
S02,113.4,30,WARN
S03,67.6,62,OK
S04,abc,41,CRITICAL
S05,72.0,,OK
S06,200.0,58,OK"""

SCHEMA = {
    "fields": {
        "sensor_id":  {"type": str,   "required": True},
        "temp_f":     {"type": float, "required": True, "min": -40, "max": 150},
        "humidity":   {"type": float, "required": False, "min": 0, "max": 100},
        "status":     {"type": str,   "required": True, "allowed": ["OK", "WARN", "CRITICAL"]},
    }
}

reader = csv.DictReader(io.StringIO(input_csv))
valid = []
rejected = []

# ── STEP 2: VALIDATE ──
for row in reader:
    errors = validate_record(row, SCHEMA)
    if errors:
        rejected.append({"row": row, "errors": errors})
    else:
        valid.append(row)

print(f"Read: 6 rows → Valid: {len(valid)}, Rejected: {len(rejected)}")
for r in rejected:
    print(f"  ❌ {r['row'].get('sensor_id','?')}: {r['errors']}")

# ── STEP 3: TRANSFORM ──
transformed = []
for row in valid:
    temp_c = round((float(row["temp_f"]) - 32) * 5/9, 1)
    transformed.append({
        "sensor_id": row["sensor_id"],
        "temp_c": temp_c,
        "humidity": row.get("humidity", "N/A"),
        "status": row["status"],
        "alert": temp_c > 35
    })

# ── STEP 4: WRITE CSV ──
csv_out = io.StringIO()
writer = csv.DictWriter(csv_out, fieldnames=["sensor_id", "temp_c", "humidity", "status", "alert"])
writer.writeheader()
writer.writerows(transformed)
print("\n── Output CSV ──")
print(csv_out.getvalue())

# ── STEP 4b: WRITE JSON ──
print("── Output JSON ──")
print(json.dumps(transformed, indent=2))

**Figure 8.2** — Pipeline with aggregation and JSON output

In [ ]:
import csv, json, io

input_csv = """student_id,name,dept,midterm,final
20230101,Zeynep Kaya,EE,78,85
20230102,Ali Demir,ME,65,72
20230103,Fatma Celik,CS,92,95
20230104,Burak Ozkan,EE,70,68
20230105,Elif Yildiz,ME,88,91
20230106,Mehmet Kara,CS,55,60"""

# Read and convert types
reader = csv.DictReader(io.StringIO(input_csv))
students = []
for row in reader:
    row["midterm"] = int(row["midterm"])
    row["final"] = int(row["final"])
    row["average"] = round(row["midterm"] * 0.4 + row["final"] * 0.6, 1)
    students.append(row)

# Aggregate by department
dept_stats = {}
for s in students:
    d = s["dept"]
    if d not in dept_stats:
        dept_stats[d] = {"students": [], "total": 0, "count": 0}
    dept_stats[d]["students"].append(s["name"])
    dept_stats[d]["total"] += s["average"]
    dept_stats[d]["count"] += 1

for d in dept_stats:
    dept_stats[d]["dept_average"] = round(dept_stats[d]["total"] / dept_stats[d]["count"], 1)
    del dept_stats[d]["total"]

# Output as JSON
report = {
    "report_title": "Student Grade Summary",
    "total_students": len(students),
    "departments": dept_stats
}

print(json.dumps(report, indent=2, ensure_ascii=False))

---
## Exercises

### Exercise 1: Read CSV Sensor Data

Parse the following CSV string using `csv.DictReader` and `io.StringIO`: `"sensor_id,temp,humidity\nS01,22.5,55\nS02,28.3,42\nS03,19.7,61"`. Print each row as a formatted string showing sensor ID, temperature, and humidity.

**Expected output:** S01: temp=22.5, humidity=55 etc.

<details><summary>💡 Hint</summary>

Use `csv.DictReader(io.StringIO(csv_text))` to iterate over rows.

</details>

In [ ]:
# ✏️ [EX1]


### Exercise 2: CSV to List of Dicts

Read this CSV: `"name,dept,score\nAhmet Yilmaz,EE,85\nAyse Demir,ME,72\nBurak Kaya,CS,91"` into a list of dicts with `score` converted to `int`. Print the list and the average score.

**Expected output:** Average score: 82.7

<details><summary>💡 Hint</summary>

Convert with `int(row['score'])` inside the loop.

</details>

In [ ]:
# ✏️ [EX2]


### Exercise 3: Write CSV from Dicts

Create a list: `data = [{'part':'resistor','count':450,'unit_price':0.05},{'part':'LED','count':300,'unit_price':0.12},{'part':'capacitor','count':120,'unit_price':0.08}]`. Write it as CSV using `csv.DictWriter` to a `StringIO`. Print the result.

**Expected output:** CSV with header: part,count,unit_price and 3 data rows.

<details><summary>💡 Hint</summary>

Set `fieldnames=['part','count','unit_price']` and call `writeheader()`.

</details>

In [ ]:
# ✏️ [EX3]


### Exercise 4: Parse JSON String

Parse this JSON: `'{"lab":"Lab-A","sensors":[{"id":"S01","temp":23.5},{"id":"S02","temp":19.8}]}'`. Print the lab name and each sensor's data.

**Expected output:** Lab: Lab-A, S01: 23.5, S02: 19.8

<details><summary>💡 Hint</summary>

Use `json.loads()` and loop over `data['sensors']`.

</details>

In [ ]:
# ✏️ [EX4]


### Exercise 5: Nested JSON Extraction

Parse: `'{"university":"ISTUN","departments":{"EE":{"students":120,"labs":5},"ME":{"students":95,"labs":4},"CS":{"students":150,"labs":6}}}'`. Print each department with its student count and number of labs.

**Expected output:** EE: 120 students, 5 labs etc.

<details><summary>💡 Hint</summary>

Loop over `data['departments'].items()`.

</details>

In [ ]:
# ✏️ [EX5]


### Exercise 6: Write JSON with Turkish Chars

Create a list of 3 student dicts with Turkish names (use ç, ş, ğ, ö, ü, ı, İ). Serialize to JSON with `indent=2` and `ensure_ascii=False`. Print the result.

**Expected output:** JSON output should show Turkish characters correctly.

<details><summary>💡 Hint</summary>

Use `json.dumps(data, indent=2, ensure_ascii=False)`.

</details>

In [ ]:
# ✏️ [EX6]


### Exercise 7: CSV Round-Trip

Start with this CSV: `"item,qty,price\nKalem,50,2.5\nSilgi,30,1.0\nDefter,20,5.0"`. Read it, add a `total` column (qty*price), and write a new CSV. Print both input and output.

**Expected output:** Output CSV should have item,qty,price,total with computed totals.

<details><summary>💡 Hint</summary>

Convert qty and price to float, compute total, add to row dict.

</details>

In [ ]:
# ✏️ [EX7]


### Exercise 8: JSON to CSV Conversion

Convert this JSON to CSV: `'[{"name":"Fatma","dept":"EE","gpa":3.5},{"name":"Ali","dept":"ME","gpa":2.8},{"name":"Elif","dept":"CS","gpa":3.9}]'`. Print the resulting CSV.

**Expected output:** CSV with header name,dept,gpa and 3 rows.

<details><summary>💡 Hint</summary>

Parse JSON first, then use `csv.DictWriter` with fieldnames from dict keys.

</details>

In [ ]:
# ✏️ [EX8]


### Exercise 9: Simple Schema Check

Define a schema requiring fields `name` (str, min_length 2), `age` (float, min 18, max 65), `dept` (str, allowed ['EE','ME','CS']). Validate these records: `{'name':'A','age':15,'dept':'XX'}` and `{'name':'Zeynep','age':22,'dept':'EE'}`. Print errors.

**Expected output:** First record should have 3 errors, second should be valid.

<details><summary>💡 Hint</summary>

Reuse the `validate_record()` function from the lecture.

</details>

In [ ]:
# ✏️ [EX9]


### Exercise 10: Validate CSV Data

Read this CSV: `"id,temp,status\nS01,23.5,OK\nS02,abc,WARN\nS03,-60,OK\nS04,22.1,BAD"`. Validate each row: temp must be float in [-40,80], status must be in [OK,WARN,CRITICAL]. Print valid and invalid rows.

**Expected output:** S01 valid, S02 invalid (non-numeric), S03 invalid (out of range), S04 invalid (bad status).

<details><summary>💡 Hint</summary>

Try `float(row['temp'])` in a try/except to catch conversion errors.

</details>

In [ ]:
# ✏️ [EX10]


### Exercise 11: CSV Filter Pipeline

Read this CSV: `"name,dept,score\nZeynep,EE,88\nAli,ME,55\nFatma,CS,92\nBurak,EE,70\nElif,ME,45\nMehmet,CS,78"`. Filter to keep only students with score >= 60. Write filtered data as new CSV. Print count before and count after.

**Expected output:** Before: 6, After: 4 (Zeynep, Fatma, Burak, Mehmet pass).

<details><summary>💡 Hint</summary>

Append to `valid` list only if `int(row['score']) >= 60`.

</details>

In [ ]:
# ✏️ [EX11]


### Exercise 12: JSON Aggregation

Parse this JSON: `'[{"sensor":"S01","readings":[22,23,21]},{"sensor":"S02","readings":[30,32,28]},{"sensor":"S03","readings":[18,19,20]}]'`. Compute min, max, avg for each sensor. Output as formatted JSON.

**Expected output:** JSON with sensor stats: S01 avg=22.0, S02 avg=30.0 etc.

<details><summary>💡 Hint</summary>

Use `sum(r)/len(r)` for avg, `min(r)` and `max(r)` for bounds.

</details>

In [ ]:
# ✏️ [EX12]


### Exercise 13: Multi-Format Output

Read CSV input: `"product,qty,price\nWidget,100,5.50\nGadget,50,12.00\nTool,200,3.25"`. Compute `total = qty * price` for each row. Output results as BOTH CSV and JSON. Print both.

**Expected output:** CSV and JSON should each show product, qty, price, and total.

<details><summary>💡 Hint</summary>

Maintain a list of dicts, then write to both CSV and JSON StringIO.

</details>

In [ ]:
# ✏️ [EX13]


### Exercise 14: Error Log Parser

Parse this JSON log: `'[{"ts":"10:01","level":"INFO","msg":"Start"},{"ts":"10:02","level":"ERROR","msg":"Sensor fail"},{"ts":"10:03","level":"WARN","msg":"Low battery"},{"ts":"10:04","level":"ERROR","msg":"Timeout"},{"ts":"10:05","level":"INFO","msg":"Retry"}]'`. Count occurrences of each level and list all ERROR messages.

**Expected output:** INFO: 2, ERROR: 2, WARN: 1. Errors: Sensor fail, Timeout.

<details><summary>💡 Hint</summary>

Use the counter pattern on `entry['level']` and filter where `level == 'ERROR'`.

</details>

In [ ]:
# ✏️ [EX14]


### Exercise 15: Full Pipeline Challenge

Build a complete pipeline: Read this CSV: `"student_id,name,dept,hw1,hw2,hw3,midterm,final\n101,Zeynep,EE,85,90,78,80,88\n102,Ali,ME,60,55,70,50,62\n103,Fatma,CS,95,92,88,90,95\n104,Burak,EE,40,45,50,35,42\n105,Elif,ME,78,82,80,75,85"`. Validate scores are 0-100. Compute weighted average (hw_avg*20% + midterm*30% + final*50%). Classify: A(>=85), B(>=70), C(>=60), F(<60). Write a JSON report with per-student results and class statistics.

**Expected output:** JSON report with 5 students, their averages, grades, and class average.

<details><summary>💡 Hint</summary>

Build it step by step: read → validate → compute → classify → aggregate → JSON output.

</details>

In [ ]:
# ✏️ [EX15]


---
### 🌉 Bridge to Next Week

You can now **read** CSV and JSON data, **validate** it against a schema, **transform** it, and **write** results back. But real-world data is rarely clean — it arrives with missing values, outliers, inconsistent types, and formatting errors. Next week you will learn to **clean and normalize** messy data, adding a critical processing stage between reading and analysis.

---
## 📮 Submission

Follow the two steps below to submit your work.

**STEP 1:** Fill in your student information and run the cell.  
**STEP 2:** Run the submission cell to send your answers.